In [ ]:
You are a medical document information extraction system.

You are given the FULL OCR text of a diagnostic lab report.
Extract structured information and return ONLY valid JSON in the exact schema below.

====================
REQUIRED OUTPUT SCHEMA
====================
{
  "patient_information": {
    "patient_name": string | null,
    "age_years": number | null,
    "gender": string | null,
    "lab_number": string | null,
    "report_status": string | null,
    "collection_datetime": string | null,
    "report_datetime": string | null,
    "laboratory_name": string | null
  },

  "tests_index": {
    "<canonical_test_key>": {
      "test_name": string,
      "value": number | string | null,
      "units": string | null,
      "reference_range": string | null,
      "interpretation": "low" | "normal" | "high" | "borderline" | null,
      "category": string
    }
  },

  "tests_by_category": {
    "complete_blood_count": [ "<canonical_test_key>", ... ],
    "liver_function": [ "<canonical_test_key>", ... ],
    "kidney_function": [ "<canonical_test_key>", ... ],
    "lipid_profile": [ "<canonical_test_key>", ... ],
    "thyroid_profile": [ "<canonical_test_key>", ... ],
    "diabetes_related": [ "<canonical_test_key>", ... ],
    "vitamins_and_minerals": [ "<canonical_test_key>", ... ],
    "electrolytes": [ "<canonical_test_key>", ... ],
    "other_tests": [ "<canonical_test_key>", ... ]
  },

  "abnormal_findings": [
    {
      "canonical_test_key": string,
      "observed_value": number | string,
      "expected_range": string,
      "severity": "low" | "high" | "critical"
    }
  ],

  "clinical_notes": {
    "interpretations": [ string ],
    "comments": [ string ],
    "notes": [ string ],
    "recommendations": [ string ],
    "disclaimers": [ string ]
  },

  "metadata": {
    "total_pages": number,
    "page_numbers_detected": [ number ],
    "report_end_marker_present": boolean
  }
}

====================
CANONICAL TEST KEY RULES
====================
- Each test MUST appear exactly ONCE in `tests_index`
- Use a normalized snake_case key derived from the test name
- Remove units, methods, symbols, and punctuation
- Examples:
  - "Hemoglobin (Photometry)" → "hemoglobin"
  - "Calcium, Total (Arsenazo III)" → "calcium_total"
  - "HbA1c" → "hba1c"
  - "Vitamin D, 25 - Hydroxy, Serum" → "vitamin_d_25_hydroxy"
- The same canonical key MUST be used everywhere

====================
INTERPRETATION RULES
====================
- Determine interpretation using reference ranges when available
- If value < lower bound → "low"
- If value > upper bound → "high"
- If borderline per report text → "borderline"
- Otherwise → "normal"

====================
STRICT RULES
====================
- Return ONLY valid JSON
- No explanations
- No markdown
- No comments
- No duplicated tests
- If information is missing, use null
- units need to be taken from reference range if mentioned or else search for it 
"""


In [12]:
# import libraries
from dotenv import load_dotenv
import os
import os
import base64
from pathlib import Path
from typing import List,Dict
from PIL import Image
from groq import Groq
import pypdfium2 as pdfium
import io
import json
from concurrent.futures import ThreadPoolExecutor

In [13]:
load_dotenv()  # loads .env from project root

True

In [14]:
if not os.getenv("GROQ_API_KEY"):
    raise RuntimeError("GROQ_API_KEY not set")

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [4]:
#creating a function to load and render pdf pages to PIL images
def load_document(file_path: str):
    ext = Path(file_path).suffix.lower()  # getting the file extension

    if ext == ".pdf":
        pdf = pdfium.PdfDocument(file_path) # loading the pdf document

        # render all pages
        # 
        images = []
        for i in range(len(pdf)):
            page = pdf[i]  # Access the specific page
            page_bitmap = page.render(scale=300/72, rotation=0) # rendering the page
            
            pil_image = page_bitmap.to_pil() # converting the page to PIL image
            images.append(pil_image)
            
            # Optional: close the page to free memory
            page.close()

        return images

    if ext in [".jpg", ".jpeg", ".png"]:
        return [Image.open(file_path)]

    raise ValueError("Unsupported file type")

In [15]:
# example of loading a pdf document
images=load_document(r"C:\Users\asuna\Downloads\sample.pdf")
print(len(images))

10


In [6]:
# helper function to encode PIL image to base64 for LLM input
def encode_pil_image(image: Image.Image) -> str:
    buffer = io.BytesIO()
    image.save(buffer, format="JPEG")
    return base64.b64encode(buffer.getvalue()).decode("utf-8")

In [ ]:
def encode_pil_image(image: Image.Image) -> str:
    """
    Encode a PIL image to a base64 string.
    LLMs (like Llama Vision) cannot 'see' a file on your hard drive.
    They need the image converted into a long text string (base64).
    """
    buffer = io.BytesIO()
    
    # Convert to RGB (remove transparency) because JPEG doesn't support transparency
    if image.mode != 'RGB':
        image = image.convert('RGB')
    
    # Save image to memory buffer as a high-quality JPEG
    image.save(buffer, format="JPEG", quality=95)
    
    # Convert binary data -> Base64 String -> UTF-8 String
    return base64.b64encode(buffer.getvalue()).decode("utf-8")

In [7]:
# function to extract entire data from single image

OCR_PROMPT = """
Act as an expert OCR engine. Transcribe the provided image into highly accurate Markdown.
- Preserve all tables using Markdown table syntax.
- Maintain the visual hierarchy (headings, sub-headings).
- Do not summarize; transcribe every word, including small print and legal clauses.
- Return ONLY the transcription.
"""

def get_markdown_from_page(image: Image.Image) -> str:   # helper function to get markdown from page
    base64_image = encode_pil_image(image)
    
    
    response = client.chat.completions.create(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": OCR_PROMPT},
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}
                    }
                ]
            }
        ],
    )
    return response.choices[0].message.content # return the response

In [8]:
# function to converted extracted text to a structured json format
EXTRACTION_PROMPT = """
You are a medical document information extraction system.

You are given the FULL OCR text of a diagnostic lab report.
Extract structured information and return ONLY valid JSON in the exact schema below.

====================
REQUIRED OUTPUT SCHEMA
====================
{
  "patient_information": {
    "patient_name": string | null,
    "age_years": number | null,
    "gender": string | null,
    "lab_number": string | null,
    "report_status": string | null,
    "collection_datetime": string | null,
    "report_datetime": string | null,
    "laboratory_name": string | null
  },

  "tests_index": {
    "<canonical_test_key>": {
      "test_name": string,
      "value": number | string | null,
      "units": string | null,
      "reference_range": string | null,
      "interpretation": "low" | "normal" | "high" | "borderline" | null,
      "category": string
    }
  },

  "tests_by_category": {
    "complete_blood_count": [ "<canonical_test_key>", ... ],
    "liver_function": [ "<canonical_test_key>", ... ],
    "kidney_function": [ "<canonical_test_key>", ... ],
    "lipid_profile": [ "<canonical_test_key>", ... ],
    "thyroid_profile": [ "<canonical_test_key>", ... ],
    "diabetes_related": [ "<canonical_test_key>", ... ],
    "vitamins_and_minerals": [ "<canonical_test_key>", ... ],
    "electrolytes": [ "<canonical_test_key>", ... ],
    "other_tests": [ "<canonical_test_key>", ... ]
  },

  "abnormal_findings": [
    {
      "canonical_test_key": string,
      "observed_value": number | string,
      "expected_range": string,
      "severity": "low" | "high" | "critical"
    }
  ],

  "clinical_notes": {
    "interpretations": [ string ],
    "comments": [ string ],
    "notes": [ string ],
    "recommendations": [ string ],
    "disclaimers": [ string ]
  },

  "metadata": {
    "total_pages": number,
    "page_numbers_detected": [ number ],
    "report_end_marker_present": boolean
  }
}

====================
CANONICAL TEST KEY RULES
====================
- Each test MUST appear exactly ONCE in `tests_index`
- Use a normalized snake_case key derived from the test name
- Remove units, methods, symbols, and punctuation
- Examples:
  - "Hemoglobin (Photometry)" → "hemoglobin"
  - "Calcium, Total (Arsenazo III)" → "calcium_total"
  - "HbA1c" → "hba1c"
  - "Vitamin D, 25 - Hydroxy, Serum" → "vitamin_d_25_hydroxy"
- The same canonical key MUST be used everywhere

====================
INTERPRETATION RULES
====================
- Determine interpretation using reference ranges when available
- If value < lower bound → "low"
- If value > upper bound → "high"
- If borderline per report text → "borderline"
- Otherwise → "normal"

====================
STRICT RULES
====================
- Return ONLY valid JSON
- No explanations
- No markdown
- No comments
- No duplicated tests
- If information is missing, use null
- units need to be taken from reference range if mentioned or else search for it 
"""

def extract_fields_from_full_text(full_contract_markdown: str) -> Dict:
    # We use a non-vision model (or the same one) for pure text-to-JSON
    response = client.chat.completions.create(
        model="moonshotai/kimi-k2-instruct",
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": "You are a diet plan and medical report analyser. Analyze the contract and return valid JSON."},
            {"role": "user", "content": f"{EXTRACTION_PROMPT}\n\nContract Text:\n{full_contract_markdown}"}
        ],
    )
    
    # Cleaning as discussed previously
    content = response.choices[0].message.content
    content = content.replace("```json", "").replace("```", "").strip()
    return json.loads(content)

In [9]:
# fucntion which combines combining raw text from individual images and creating a json file with the total extracted text
def process_document_with_llm_ocr(file_path: str):
    images = load_document(file_path)
    full_document_markdown = ""

    # Step 1: LLM-OCR for every page
    for i, img in enumerate(images, start=1):
        page_text = get_markdown_from_page(img)
        full_document_markdown += f"\n--- PAGE {i} ---\n{page_text}"

    # Step 2: Extract JSON from the unified text
    final_data = extract_fields_from_full_text(full_document_markdown)
    
    return final_data

In [10]:
# Execution
final_result = process_document_with_llm_ocr(r"C:\Users\asuna\Downloads\sample.pdf")
print(json.dumps(final_result, indent=2))

{
  "patient_information": {
    "patient_name": "DUMMY",
    "age_years": 30,
    "gender": "Male",
    "lab_number": "439854467",
    "report_status": "Final",
    "collection_datetime": "14/5/2023 11:03:00AM",
    "report_datetime": "16/5/2023 1:36:25PM",
    "laboratory_name": "Dr Lal PathLabs Ltd., Block-E, Sector-18, Rohini, New Delhi-110085"
  },
  "tests_index": {
    "hemoglobin": {
      "test_name": "Hemoglobin",
      "value": 15.0,
      "units": "g/dL",
      "reference_range": "13.00 - 17.00",
      "interpretation": "normal",
      "category": "complete_blood_count"
    },
    "packed_cell_volume": {
      "test_name": "Packed Cell Volume (PCV)",
      "value": 45.0,
      "units": "%",
      "reference_range": "40.00 - 50.00",
      "interpretation": "normal",
      "category": "complete_blood_count"
    },
    "rbc_count": {
      "test_name": "RBC Count",
      "value": 4.5,
      "units": "mill/mm3",
      "reference_range": "4.50 - 5.50",
      "interpretation": "n

In [11]:
# combining ocr and extraction using thread pool
def process_document_with_llm_ocr_v2(file_path: str):
    images = load_document(file_path)

    # Step 1: OCR all pages in parallel
    workers=min(8,len(images))
    with ThreadPoolExecutor(max_workers=workers) as executor:
        page_texts = list(executor.map(get_markdown_from_page, images))

    # Step 2: Combine OCR results in order
    chunks=[]
    for i, page_text in enumerate(page_texts, start=1):
        chunks.append(f"\n--- PAGE {i} ---\n{page_text}")
    full_document_markdown = "".join(chunks)

    # Step 3: Extract structured data
    final_data = extract_fields_from_full_text(full_document_markdown)

    return final_data

# Execution
final_result = process_document_with_llm_ocr_v2(r"C:\Users\asuna\Downloads\sample.pdf")
print(json.dumps(final_result, indent=2))


{
  "patient_information": {
    "patient_name": "DUMMY",
    "age_years": 30,
    "gender": "Male",
    "lab_number": "439854467",
    "report_status": "Final",
    "collection_datetime": "14/5/2023 11:03:00AM",
    "report_datetime": "16/5/2023 1:36:25PM",
    "laboratory_name": "Dr Lal PathLabs Ltd"
  },
  "tests_index": {
    "hemoglobin": {
      "test_name": "Hemoglobin (Photometry)",
      "value": 15.0,
      "units": "g/dL",
      "reference_range": "13.00 - 17.00",
      "interpretation": "normal",
      "category": "complete_blood_count"
    },
    "packed_cell_volume": {
      "test_name": "Packed Cell Volume (PCV) (Calculated)",
      "value": 45.0,
      "units": "%",
      "reference_range": "40.00 - 50.00",
      "interpretation": "normal",
      "category": "complete_blood_count"
    },
    "rbc_count": {
      "test_name": "RBC Count (Electrical Impedence)",
      "value": 4.5,
      "units": "mill/mm3",
      "reference_range": "4.50 - 5.50",
      "interpretation": 